# 09: ablacje: czy potrzebna jest kamera B? czy potrzebna jest geometria?

Bez nowego treningu i bez RAFT: tylko modele już wytrenowane na `rs_video_x4k` (ten sam zbiór, ta sama walidacja,
321 próbek ze scen spoza treningu). **Wystarczy środowisko bez GPU** (Środowisko wykonawcze → Zmień typ → CPU, nie zużywa
jednostek); sekcja 5 (opcjonalna, model główny z RAFT) idzie szybciej na GPU.

**A. Kamera B.** Dwa modele różniące się tylko tym, czy widzą obraz z kamery B (zwykły U-Net od zera, 200 epok, bs 16,
lr 3e-4; bez B = sieć dostaje kopię obrazu A, architektura i liczba wag bez zmian):

| model | run | psnr_full |
|---|---|---|
| U-Net, A + B | `2026-11-11_unet_plain_long` | 30,94 |
| U-Net, tylko A | `2026-11-11_unet_plain_single` | 26,51 |
| bez korekcji | – | 26,35 |

Sama różnica nie mówi, **co** B daje. Kamera A czyta wiersze, B kolumny, więc dla piksela p chwile odczytu to
`t_A(p)` i `t_B(p)`. Na przekątnej (prawy górny → lewy dolny róg) `t_B = t_A`: obie kamery czytają punkt w tej samej chwili
i B nie niesie żadnej informacji o ruchu; im dalej od przekątnej, tym większe `|t_B − t_A|` (do ~30 ms). **Jeśli zysk
z kamery B rośnie z `|t_B − t_A|`, B pomaga dzięki innej chwili odczytu** (to jest pomysł pracy: prostopadły odczyt),
a nie dzięki dodatkowym pikselom czy drugiemu punktowi widzenia. Sekcja 4 liczy zysk osobno dla każdego przedziału
`|t_B − t_A|` i (bo błąd wejścia zależy od odległości od środkowego wiersza) dla każdego przedziału `|t_A|`.

**B. Geometria.** Odpowiedź jest w istniejących runach (sekcja 3): RAFT + geometria bez żadnego uczenia (32,86 dB, notatnik 04)
jest lepszy od najlepszej syntezy wprost (`corr` 31,06 dB, zwykły U-Net 30,94 dB, autoenkoder 23,27 dB), a z dostrojeniem
i U-Netem 34,30 / 35,81 dB.

**Porzucone (14.11):** `corr` bez kamery B i `corrgeom` (sieć `corr` przewidująca przepływ + geometria). Sieć `corr` nie ma
warstw normalizacji i przy lr 3e-4 rozjeżdżała się po szczycie lr (fp16: NaN, bf16: wyjście nasycone na czarny obraz,
`corrgeom`: przepływ nasycony); 4 próby, opis w `experiments/README.md`. Wniosek do pracy: dopasowanie A↔B uczone od zera
samą stratą obrazu nie zbiega, potrzebny jest dobry estymator przepływu.

**Kolejność:** SETUP → 1 → 2 → 3 → 4 (→ 5 opcjonalnie). Wyniki: `MyDrive/rs_project/results/ablacja_kamery_b/`.

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Modele
Najnowsze runy o danej końcówce nazwy z `MyDrive/rs_project/runs/`.

In [ ]:
import torch
from raft_unet import load_model
from raft_geom import RSSampleDataset
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RUN_AB, RUN_A = P.find_run('unet_plain_long'), P.find_run('unet_plain_single')
assert RUN_AB and RUN_A, f'brak runów: A+B {RUN_AB}, A {RUN_A}'
MODELS = {'A+B': load_model(RUN_AB, DEV)[0], 'A': load_model(RUN_A, DEV)[0]}
VAL = RSSampleDataset(f'{DATA}/val')
OUT = P.RESULTS / 'ablacja_kamery_b'
OUT.mkdir(parents=True, exist_ok=True)
print('urządzenie:', DEV, '| A+B:', RUN_AB.parent.name, '| A:', RUN_A.parent.name, '|', len(VAL), 'próbek val')

## 3. Wynik ogólny (A i B)
Z plików `results.json` / `eval.json` runów. `psnr` = obszar widoczny w A (jak w 04), `psnr_full` = cały obraz
(wspólna miara; dla RAFT z 04 jest tylko `psnr`).

In [ ]:
import json, pandas as pd
def metrics(rj):
    m = json.load(open(rj))
    return m.get('val_metrics') or m.get('metrics') or m
rows = []
def add(question, label, rj):
    if rj and rj.exists():
        m = metrics(rj)
        rows.append({'pytanie': question, 'model': label, 'run': rj.parent.name, 'psnr': m.get('psnr'),
                     'psnr_full': m.get('psnr_full'), 'ssim': m.get('ssim'), 'wejście (full)': m.get('psnr_full_in')})
    else:
        print('brak', label, rj)
add('A: kamera B', 'zwykły U-Net, A + B, 200 ep.', P.find_run('unet_plain_long', 'results.json'))
add('A: kamera B', 'zwykły U-Net, tylko A, 200 ep.', P.find_run('unet_plain_single', 'results.json'))
add('B: geometria', 'autoenkoder bez skipów (ae), 60 ep.', P.find_run('ae', 'results.json'))
add('B: geometria', 'zwykły U-Net (bez geometrii), 200 ep.', P.find_run('unet_plain_long', 'results.json'))
add('B: geometria', 'enkoder + korelacja (corr, bez geometrii), 80 ep.', P.find_run('corr', 'results.json'))
add('B: geometria', 'RAFT gotowy + geometria v1 (04, bez uczenia)', P.find_run('raft_zeroshot', 'eval.json'))
add('B: geometria', 'RAFT dostrojony + geometria v1 (04, 10 ep.)', P.find_run('raft_geom', 'results.json'))
add('B: geometria', 'RAFT + geometria + U-Net, wspólnie dostrojone (05)', P.find_run('raft_unet_ft', 'results.json'))
SUMMARY = pd.DataFrame(rows).set_index(['pytanie', 'model'])
display(SUMMARY.round(3))
SUMMARY.to_csv(OUT / 'podsumowanie.csv')

## 4. A: gdzie pomaga kamera B (zysk w funkcji różnicy chwil odczytu)
Dla każdego piksela każdej próbki val: błąd kwadratowy modelu A + B, modelu tylko A i wejścia, pogrupowany według
`|t_A|` (ile jest do poprawienia: środkowy wiersz `t_A = 0` jest prawie bez zniekształceń) i `|t_B − t_A|` (ile informacji
o ruchu daje B). Czas `t_B` brany w tym samym miejscu obrazu (przesunięcie między kamerami, do ~40 px, pomijamy).
Na CPU ~2–5 min; wynik zapisuje się w `OUT`, ponowne uruchomienie go wczytuje (`REDO = True` liczy od nowa).

Jak czytać:
* tabela: wiersze = przedział `|t_A|` i rodzaj zysku, kolumny = przedział `|t_B − t_A|`; `A+B - A` = zysk z kamery B,
* wykres po lewej: zysk z kamery B w funkcji `|t_B − t_A|`; **rosnąca krzywa = B pomaga dzięki innej chwili odczytu**,
* środek: średnie `|t_B − t_A|` w każdym miejscu obrazu (ciemny pas = obie kamery czytają jednocześnie),
* po prawej: zysk z kamery B w każdym miejscu obrazu; jeśli hipoteza jest prawdziwa, wzór przypomina mapę ze środka.

In [ ]:
import pickle
import analysis_b as ab
REDO = False
cache = OUT / 'analiza.pkl'
if cache.exists() and not REDO:
    RES = pickle.load(open(cache, 'rb'))
else:
    RES = ab.analyze(MODELS, VAL, DEV, bs=8, workers=WORKERS)
    pickle.dump(RES, open(cache, 'wb'))
TAB = ab.gain_table(RES, 'A+B', 'A')
display(TAB.round(2))
TAB.to_csv(OUT / 'zysk_z_kamery_b.csv')
c, g, gab, ga, n = ab.gain_by_dt(RES, 'A+B', 'A', min_ta_bin=1)
print('|t_A| >= 5 ms (jest co poprawiać), zysk z kamery B w kolejnych przedziałach |t_B - t_A|:')
for ci, gi, gabi, gai, ni in zip(c, g, gab, ga, n):
    print(f'  |t_B - t_A| ~{ci:4.1f} ms: A+B - A {gi:+.2f} dB  (A+B - wejście {gabi:+.2f}, A - wejście {gai:+.2f})  '
          f'{100 * ni / n.sum():4.1f}% pikseli')
print(f'korelacja (po położeniu w obrazie) zysku z kamery B i |t_B - t_A|: {ab.correlation_gain_dt(RES, "A+B", "A"):+.2f}')
fig = ab.plot_gain(RES, 'A+B', 'A', OUT / 'zysk_z_kamery_b.png')
print('zapisano', OUT / 'zysk_z_kamery_b.png', 'i', OUT / 'zysk_z_kamery_b.csv')

## 5. (opcjonalnie) To samo dla modelu głównego (RAFT + geometria + U-Net, notatnik 05)
Model główny nie ma wersji bez kamery B (geometria jej wymaga), więc tu: zysk względem wejścia w funkcji `|t_B − t_A|`.
We wzorze geometrii prędkość = przesunięcie / (`t_B − t_A`), więc przy `t_B ≈ t_A` jest nieokreślona i geometria ją
uzupełnia z sąsiedztwa; spodziewany słabszy wynik przy małym `|t_B − t_A|`. RAFT na CPU jest wolny (~10–20 min), na GPU ~1 min.

In [ ]:
RUN_MAIN = P.find_run('raft_unet_ft')
cache_main = OUT / 'analiza_model_glowny.pkl'
if cache_main.exists() and not REDO:
    RES_MAIN = pickle.load(open(cache_main, 'rb'))
else:
    RES_MAIN = ab.analyze({'główny': load_model(RUN_MAIN, DEV)[0]}, VAL, DEV, bs=4, workers=WORKERS)
    pickle.dump(RES_MAIN, open(cache_main, 'wb'))
c, g, *_ , n = ab.gain_by_dt(RES_MAIN, 'główny', 'input', min_ta_bin=1)
print(RUN_MAIN.parent.name, '- zysk względem wejścia, |t_A| >= 5 ms:')
for ci, gi, ni in zip(c, g, n):
    print(f'  |t_B - t_A| ~{ci:4.1f} ms: {gi:+.2f} dB  ({100 * ni / n.sum():4.1f}% pikseli)')

## 6. Po analizie
Dopisz do `experiments/README.md` (sekcja „Wnioski”): zysk z kamery B przy małym i dużym `|t_B − t_A|` (z sekcji 4)
i korelację; rysunek `zysk_z_kamery_b.png` skopiuj do `docs/rysunki/`. Commit notatnika z wynikami.